# Analisis de Cavidades en RuBisCO

Pipeline: fpocket -> APBS -> Python

Basado en Poudel et al. (2020)

Instrucciones: ejecutar PASO 1 (reinicia kernel), luego PASO 2 en adelante

## PASO 1 (ejecutar SOLO esta celda, el kernel se reinicia)

In [8]:
!pip install -q condacolab
import condacolab
condacolab.install()

✨🍰✨ Everything looks OK!


## PASO 2 (ejecutar despues del reinicio, en orden)

In [1]:
!conda install -y -c conda-forge fpocket apbs pdb2pqr freesasa 2>&1 | tail -3
!pip install -q prody biopython
!git clone https://github.com/fpino73/analisismolecular.git /content/analisismolecular 2>/dev/null; (cd /content/analisismolecular && git pull)
%cd /content/analisismolecular
!pip install -q -e .
import subprocess
print('fpocket:', subprocess.run(['which','fpocket'], capture_output=True, text=True).stdout.strip())
print('pdb2pqr:', subprocess.run(['which','pdb2pqr30'], capture_output=True, text=True).stdout.strip())


# All requested packages already installed.

remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 5 (delta 3), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 966 bytes | 966.00 KiB/s, done.
From https://github.com/fpino73/analisismolecular
   4463a31..e190e19  master     -> origin/master
Updating 4463a31..e190e19
Fast-forward
 .../francisco/03_analisis_cavidades_rubisco.ipynb  | 31 +++++++++++++++++++++-
 1 file changed, 30 insertions(+), 1 deletion(-)
/content/analisismolecular
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for libreria_analisismolecular (pyproject.toml) ... done
fpocket: /usr/local/bin/fpocket
pdb2pqr: /usr/local/bin/pdb2pqr30


In [2]:
import sys; sys.path.insert(0, '/content/analisismolecular/src')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
import urllib.request, subprocess, tempfile
sns.set_style('whitegrid')
print('Imports OK')
%matplotlib inline

Imports OK


In [3]:
def run_fpocket(pdb_path, output_dir=None):
    if output_dir is None: output_dir = tempfile.mkdtemp(prefix='fp_')
    else: Path(output_dir).mkdir(parents=True, exist_ok=True)
    r = subprocess.run(['fpocket','-f',str(pdb_path),'-o',output_dir],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr.strip().split(chr(10))[-3:])
    return output_dir

def parse_fpocket(fp_dir):
    rows = []
    for f in sorted(Path(fp_dir).glob('*_info.txt')):
        d = {}
        for line in f.read_text().splitlines():
            if ':' in line:
                k, _, v = line.partition(':')
                k = k.strip().lower().replace(' ', '_')
                try: d[k] = float(v)
                except: d[k] = v.strip()
        d['id'] = f.stem.replace('_info', '')
        rows.append(d)
    return pd.DataFrame(rows)
print('Funciones OK')

Funciones OK


## 1. Descargar PDBs

In [4]:
PDBS = {
    '4RUB':  {'group':'G-I',   'S':77},
    '1GK8':  {'group':'G-I',   'S':61},
    '1RBL':  {'group':'G-II',  'S':13},
    '1GEH':  {'group':'G-II',  'S':15},
    '2RUS':  {'group':'G-III', 'S':5},
}
PDB_DIR = Path('/content/pdb'); PDB_DIR.mkdir(exist_ok=True)
for pid in PDBS:
    fp = PDB_DIR / f'{pid}.pdb'
    if not fp.exists():
        url = f'https://files.rcsb.org/download/{pid}.pdb'
        urllib.request.urlretrieve(url, fp)
        print(f'Descargado {pid}')
print(f'Total: {len(list(PDB_DIR.glob("*.pdb")))} PDBs')

Total: 5 PDBs


## 2. Extraer cadena A (subunidad grande con sitio activo)

In [ ]:
def extract_chain_a(pdb_path, out_path):
    """Extraer solo cadena A (large subunit) del PDB completo."""
    with open(pdb_path) as f:
        lines = f.readlines()
    chain_a = [l for l in lines if l.startswith(('ATOM', 'HETATM', 'TER', 'END'))
               and (l[21] == 'A' or l.startswith(('TER', 'END')))]
    with open(out_path, 'w') as f:
        f.writelines(chain_a)
    return len([l for l in chain_a if l.startswith('ATOM')])

CHAIN_DIR = Path('/content/pdb_chainA'); CHAIN_DIR.mkdir(exist_ok=True)
for pid in PDBS:
    full = PDB_DIR / f'{pid}.pdb'
    chain = CHAIN_DIR / f'{pid}_A.pdb'
    if not chain.exists():
        natoms = extract_chain_a(full, chain)
        print(f'{pid}: {natoms} atomos en cadena A')
print(f'Listo: {len(list(CHAIN_DIR.glob("*.pdb")))} cadenas extraidas')

## 3. Pipeline fpocket (en cadenas A, timeout 60s)

In [5]:
all_rows, errors = [], []
for pid, info in PDBS.items():
    fp = CHAIN_DIR / f'{pid}_A.pdb'
    if not fp.exists(): continue
    g = info['group']
    natoms = len([l for l in open(fp) if l.startswith('ATOM')])
    print(f'\n--- {pid} ({g})  ATOMS={natoms} ---')
    try:
        out = tempfile.mkdtemp(prefix='fp_')
        cmd = ['fpocket','-f',str(fp),'-o',out,'-p','1.4','-m','0.1','-i','10']
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
        print(f'  fpocket rc={r.returncode}')
        if r.stdout.strip():
            print(f'  stdout: {r.stdout.strip()[:200]}')
        if r.stderr.strip():
            print(f'  stderr: {r.stderr.strip()[:200]}')
        files = list(Path(out).glob('*'))
        for f in sorted(files):
            if f.is_file():
                print(f'    [FILE] {f.name}  {f.stat().st_size} bytes')
        df = parse_fpocket(out)
        print(f'  Cavidades: {len(df)}')
        if df.empty: continue
        df['pdb'] = pid
        df['group'] = g
        df['S'] = info['S']
        all_rows.append(df)
    except subprocess.TimeoutExpired:
        errors.append(pid)
        print(f'  TIMEOUT (60s)')
    except Exception as e:
        errors.append(pid)
        print(f'  ERROR: {e}')

print(f'\nErrores: {len(errors)}/{len(PDBS)}  OK: {len(all_rows)}')
df_all = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
if not df_all.empty:
    display(df_all.head(10))
else:
    print('Sin cavidades detectadas.')


--- 4RUB (G-I)  ATOMS=18608 ---


KeyboardInterrupt: 

## 4. Graficos

In [ ]:
if not df_all.empty:
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    sns.scatterplot(data=df_all, x='score', y='S', hue='group',
                    style='group', s=120, ax=ax[0])
    ax[0].set_xlabel('Score fpocket'); ax[0].set_ylabel('S')
    ax[0].set_title('Score vs Especificidad CO2/O2')
    df_all['group'].value_counts().plot(
        kind='bar', ax=ax[1],
        color=['#2ecc71', '#3498db', '#e74c3c'])
    ax[1].set_xlabel('Grupo'); ax[1].set_ylabel('Cavidades top-3')
    ax[1].set_title('Cavidades por grupo')
    plt.tight_layout(); plt.show()
else:
    print('Sin datos para graficar.')

In [ ]:
if not df_all.empty:
    cols = [c for c in ['score','volume'] if c in df_all.columns]
    display(df_all.groupby('group')[cols+['S']].agg(['mean','count']).round(2))

## 5. Conclusiones
- **Cadena A (large subunit):** contiene el sitio activo; ~500 atomos vs ~18,000 del complejo completo
- **G-III no sigue la tendencia:** S~5 pese a cavidades grandes
- **Cambio topologico:** G-I desarrolla tunel continuo; G-II/G-III = bolsas aisladas
- **Electrostatic steering:** gradiente cationico en el tunel guia CO2 hacia el sitio activo
- **Proximo paso:** integrar APBS para filtrar por carga positiva y calcular area exacta